# Extract content from recorder customer calls


This notebook demos how to use Azure Content Understanding (preview) to extact content from recorded calls using a custom analyzer. The analyzer is built using the Azure AI Foundry Studio to define the schema for data to be extracted. The interest in this analyzer is questions that the caller has asked in an effort to build a list of questions that can be furthered anaylzed.

## Prerequisites
1. Ensure Azure AI service is configured following [steps](../README.md#configure-azure-ai-service-resource)
2. Install the required packages to run the sample.

In [ ]:
!pip install jsonpath-ng
!pip install pandas

## Create Azure AI Content Understanding Client

> The [AzureContentUnderstandingClient](../python/content_understanding_client.py) is a utility class containing functions to interact with the Content Understanding API. Before the official release of the Content Understanding SDK, it can be regarded as a lightweight SDK.


## IMPORTANT: Set these variables in .env file

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

## Set these variables for your environment
AZURE_AI_ENDPOINT = os.environ.get("AZURE_AI_ENDPOINT")
subscription_key = os.environ.get("AZURE_AI_API_KEY")
AZURE_AI_API_VERSION = os.environ.get("AZURE_AI_API_VERSION")
ANALYZER_ID = os.environ.get("ANALYZER_ID")

## IMPORTANT: Set file paths depending on your environment

In [ ]:
# For running in Microsoft Fabric
# ANALYZER_FOLDER_INPUT = '/lakehouse/default/Files/calls/input'
# ANALYZER_FOLDER_OUTPUT = '/lakehouse/default/Files/calls/extract'
# ANALYZER_FOLDER_ERROR_LOG = '/lakehouse/default/Files/calls/error'

# For running locally
ANALYZER_FOLDER_INPUT = f'../data/calls/input'
ANALYZER_FOLDER_OUTPUT = f'../data/calls/extract'
ANALYZER_FOLDER_ERROR_LOG = f'../data/calls/error'

## IMPORTANT: DO NOT CHANGE THIS CELL!!

In [ ]:
import requests
from requests.models import Response
import logging
import json
import time
from pathlib import Path


class AzureContentUnderstandingClient:
    def __init__(
        self,
        endpoint: str,
        api_version: str,
        subscription_key: str = None,
        token_provider: callable = None,
        x_ms_useragent: str = "cu-sample-code",
    ):
        if not subscription_key and not token_provider:
            raise ValueError(
                "Either subscription key or token provider must be provided."
            )
        if not api_version:
            raise ValueError("API version must be provided.")
        if not endpoint:
            raise ValueError("Endpoint must be provided.")

        self._endpoint = endpoint.rstrip("/")
        self._api_version = api_version
        self._logger = logging.getLogger(__name__)
        self._headers = self._get_headers(
            subscription_key, token_provider() if token_provider else None, x_ms_useragent
        )

    def _get_analyzer_url(self, endpoint, api_version, analyzer_id):
        return f"{endpoint}/contentunderstanding/analyzers/{analyzer_id}?api-version={api_version}"  # noqa

    def _get_analyzer_list_url(self, endpoint, api_version):
        return f"{endpoint}/contentunderstanding/analyzers?api-version={api_version}"

    def _get_analyze_url(self, endpoint, api_version, analyzer_id):
        return f"{endpoint}/contentunderstanding/analyzers/{analyzer_id}:analyze?api-version={api_version}"  # noqa

    def _get_training_data_config(
        self, storage_container_sas_url, storage_container_path_prefix
    ):
        return {
            "containerUrl": storage_container_sas_url,
            "kind": "blob",
            "prefix": storage_container_path_prefix,
        }

    def _get_headers(self, subscription_key, api_token, x_ms_useragent):
        """Returns the headers for the HTTP requests.
        Args:
            subscription_key (str): The subscription key for the service.
            api_token (str): The API token for the service.
            enable_face_identification (bool): A flag to enable face identification.
        Returns:
            dict: A dictionary containing the headers for the HTTP requests.
        """
        headers = (
            {"Ocp-Apim-Subscription-Key": subscription_key}
            if subscription_key
            else {"Authorization": f"Bearer {api_token}"}
        )
        headers["x-ms-useragent"] = x_ms_useragent
        return headers

    def get_all_analyzers(self):
        """
        Retrieves a list of all available analyzers from the content understanding service.

        This method sends a GET request to the service endpoint to fetch the list of analyzers.
        It raises an HTTPError if the request fails.

        Returns:
            dict: A dictionary containing the JSON response from the service, which includes
                  the list of available analyzers.

        Raises:
            requests.exceptions.HTTPError: If the HTTP request returned an unsuccessful status code.
        """
        response = requests.get(
            url=self._get_analyzer_list_url(self._endpoint, self._api_version),
            headers=self._headers,
        )
        response.raise_for_status()
        return response.json()

    def get_analyzer_detail_by_id(self, analyzer_id):
        """
        Retrieves a specific analyzer detail through analyzerid from the content understanding service.
        This method sends a GET request to the service endpoint to get the analyzer detail.

        Args:
            analyzer_id (str): The unique identifier for the analyzer.

        Returns:
            dict: A dictionary containing the JSON response from the service, which includes the target analyzer detail.

        Raises:
            HTTPError: If the request fails.
        """
        response = requests.get(
            url=self._get_analyzer_url(self._endpoint, self._api_version, analyzer_id),
            headers=self._headers,
        )
        response.raise_for_status()
        return response.json()

    def begin_create_analyzer(
        self,
        analyzer_id: str,
        analyzer_template: dict = None,
        analyzer_template_path: str = "",
        training_storage_container_sas_url: str = "",
        training_storage_container_path_prefix: str = "",
    ):
        """
        Initiates the creation of an analyzer with the given ID and schema.

        Args:
            analyzer_id (str): The unique identifier for the analyzer.
            analyzer_template (dict, optional): The schema definition for the analyzer. Defaults to None.
            analyzer_template_path (str, optional): The file path to the analyzer schema JSON file. Defaults to "".
            training_storage_container_sas_url (str, optional): The SAS URL for the training storage container. Defaults to "".
            training_storage_container_path_prefix (str, optional): The path prefix within the training storage container. Defaults to "".

        Raises:
            ValueError: If neither `analyzer_template` nor `analyzer_template_path` is provided.
            requests.exceptions.HTTPError: If the HTTP request to create the analyzer fails.

        Returns:
            requests.Response: The response object from the HTTP request.
        """
        if analyzer_template_path and Path(analyzer_template_path).exists():
            with open(analyzer_template_path, "r") as file:
                analyzer_template = json.load(file)

        if not analyzer_template:
            raise ValueError("Analyzer schema must be provided.")

        if (
            training_storage_container_sas_url
            and training_storage_container_path_prefix
        ):  # noqa
            analyzer_template["trainingData"] = self._get_training_data_config(
                training_storage_container_sas_url,
                training_storage_container_path_prefix,
            )

        headers = {"Content-Type": "application/json"}
        headers.update(self._headers)

        response = requests.put(
            url=self._get_analyzer_url(self._endpoint, self._api_version, analyzer_id),
            headers=headers,
            json=analyzer_template,
        )
        response.raise_for_status()
        self._logger.info(f"Analyzer {analyzer_id} create request accepted.")
        return response

    def delete_analyzer(self, analyzer_id: str):
        """
        Deletes an analyzer with the specified analyzer ID.

        Args:
            analyzer_id (str): The ID of the analyzer to be deleted.

        Returns:
            response: The response object from the delete request.

        Raises:
            HTTPError: If the delete request fails.
        """
        response = requests.delete(
            url=self._get_analyzer_url(self._endpoint, self._api_version, analyzer_id),
            headers=self._headers,
        )
        response.raise_for_status()
        self._logger.info(f"Analyzer {analyzer_id} deleted.")
        return response

    def begin_analyze(self, analyzer_id: str, file_location: str):
        """
        Begins the analysis of a file or URL using the specified analyzer.

        Args:
            analyzer_id (str): The ID of the analyzer to use.
            file_location (str): The path to the file or the URL to analyze.

        Returns:
            Response: The response from the analysis request.

        Raises:
            ValueError: If the file location is not a valid path or URL.
            HTTPError: If the HTTP request returned an unsuccessful status code.
        """
        data = None
        if Path(file_location).exists():
            with open(file_location, "rb") as file:
                data = file.read()
            headers = {"Content-Type": "application/octet-stream"}
        elif "https://" in file_location or "http://" in file_location:
            data = {"url": file_location}
            headers = {"Content-Type": "application/json"}
        else:
            raise ValueError("File location must be a valid path or URL.")

        headers.update(self._headers)
        if isinstance(data, dict):
            response = requests.post(
                url=self._get_analyze_url(
                    self._endpoint, self._api_version, analyzer_id
                ),
                headers=headers,
                json=data,
            )
        else:
            response = requests.post(
                url=self._get_analyze_url(
                    self._endpoint, self._api_version, analyzer_id
                ),
                headers=headers,
                data=data,
            )

        response.raise_for_status()
        self._logger.info(
            f"Analyzing file {file_location} with analyzer: {analyzer_id}"
        )
        return response

    def get_image_from_analyze_operation(
        self, analyze_response: Response, image_id: str
    ):
        """Retrieves an image from the analyze operation using the image ID.
        Args:
            analyze_response (Response): The response object from the analyze operation.
            image_id (str): The ID of the image to retrieve.
        Returns:
            bytes: The image content as a byte string.
        """
        operation_location = analyze_response.headers.get("operation-location", "")
        if not operation_location:
            raise ValueError(
                "Operation location not found in the analyzer response header."
            )
        operation_location = operation_location.split("?api-version")[0]
        image_retrieval_url = (
            f"{operation_location}/images/{image_id}?api-version={self._api_version}"
        )
        try:
            response = requests.get(url=image_retrieval_url, headers=self._headers)
            response.raise_for_status()

            assert response.headers.get("Content-Type") == "image/jpeg"

            return response.content
        except requests.exceptions.RequestException as e:
            print(f"HTTP request failed: {e}")
            return None

    def poll_result(
        self,
        response: Response,
        timeout_seconds: int = 120,
        polling_interval_seconds: int = 2,
    ):
        """
        Polls the result of an asynchronous operation until it completes or times out.

        Args:
            response (Response): The initial response object containing the operation location.
            timeout_seconds (int, optional): The maximum number of seconds to wait for the operation to complete. Defaults to 120.
            polling_interval_seconds (int, optional): The number of seconds to wait between polling attempts. Defaults to 2.

        Raises:
            ValueError: If the operation location is not found in the response headers.
            TimeoutError: If the operation does not complete within the specified timeout.
            RuntimeError: If the operation fails.

        Returns:
            dict: The JSON response of the completed operation if it succeeds.
        """
        operation_location = response.headers.get("operation-location", "")
        if not operation_location:
            raise ValueError("Operation location not found in response headers.")

        headers = {"Content-Type": "application/json"}
        headers.update(self._headers)

        start_time = time.time()
        while True:
            elapsed_time = time.time() - start_time
            if elapsed_time > timeout_seconds:
                raise TimeoutError(
                    f"Operation timed out after {timeout_seconds:.2f} seconds."
                )

            response = requests.get(operation_location, headers=self._headers)
            response.raise_for_status()
            status = response.json().get("status").lower()
            if status == "succeeded":
                self._logger.info(
                    f"Request result is ready after {elapsed_time:.2f} seconds."
                )
                return response.json()
            elif status == "failed":
                self._logger.error(f"Request failed. Reason: {response.json()}")
                raise RuntimeError("Request failed.")
            else:
                self._logger.info(
                    f"Request {operation_location.split('/')[-1].split('?')[0]} in progress ..."
                )
            time.sleep(polling_interval_seconds)


In [ ]:
import logging
import json
import sys
from pathlib import Path
# from azure.identity import DefaultAzureCredential, get_bearer_token_provider, InteractiveBrowserCredential

client = AzureContentUnderstandingClient(
    endpoint=AZURE_AI_ENDPOINT,
    api_version=AZURE_AI_API_VERSION,
    subscription_key=subscription_key
)

# load_dotenv(find_dotenv())
# logging.basicConfig(level=logging.INFO)

# credential = DefaultAzureCredential()
# credential = InteractiveBrowserCredential()
# token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

# client = AzureContentUnderstandingClient(
#     endpoint=AZURE_AI_ENDPOINT,
#     api_version=AZURE_AI_API_VERSION,
#     token_provider=token_provider
# )

## Loop through /calls folder and analyze each call

In [ ]:
import posixpath
import datetime

now = datetime.datetime.now()
timestamp = now.strftime("%Y%m%d%H%M%S")
        
# Create output folder and error folder if it does not exist
os.makedirs(ANALYZER_FOLDER_OUTPUT, exist_ok=True)
os.makedirs(ANALYZER_FOLDER_ERROR_LOG, exist_ok=True)

# Loop through each file in the input folder
for filename in os.listdir(ANALYZER_FOLDER_INPUT):
    try:
        # Resolve the full path of the file and extract the filename and extension
        file_path = posixpath.join(ANALYZER_FOLDER_INPUT, filename)
        filename_root,filename_extension = os.path.splitext(filename)

        # Make API call to analyze the file. Poll for completion
        print(f"Analyzing call file: {file_path}")
        print(f"{ANALYZER_FOLDER_OUTPUT}/{filename_root}.json")
        response = client.begin_analyze(ANALYZER_ID, file_location=file_path)
        result = client.poll_result(response)
        print(json.dumps(result, indent=2))

        # Save the result to the output folder
        with open(f"{ANALYZER_FOLDER_OUTPUT}/{filename_root}.json", "w") as f:
            f.write(json.dumps(result, indent=2))
            print(f"Saved result to {ANALYZER_FOLDER_OUTPUT}/{filename_root}.json")
    
    except Exception as e:
        print(f"Error analyzing call file: {file_path}/{filename}")
        print(e)
        # write error to log file
        with open(f"{ANALYZER_FOLDER_ERROR_LOG}/error{timestamp}.log", "a") as f:
            f.write(f"Error analyzing call file: {file_path}/{filename}\n")
            f.write(str(e))
            f.write("\n")
        continue



In [ ]:
import os
import json
import pandas as pd
from jsonpath_ng import jsonpath, parse

# Define the path to the folder containing the JSON files
folder_path = ANALYZER_FOLDER_OUTPUT

# Define the JSONPath expression to extract Customer_Questions
jsonpath_expr_customer_questions = parse('$.result.contents[*].fields.Customer_Questions.valueArray[*].valueString')
jsonpath_expr_results = parse('$.result')

# Initialize an empty list to store the data
data_list = []

# Iterate over each file in the folder
for filename in os.listdir(folder_path):
    if filename.endswith('.json'):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'r') as file:
            data = json.load(file)
            # Extract the Customer_Questions field using JSONPath
            match_customer_questions = jsonpath_expr_customer_questions.find(data)
            match_results = jsonpath_expr_results.find(data)
            for match in match_customer_questions:
                row = {
                    'id': parse('$.id').find(data)[0].value,
                    'Filename': filename,
                    'analyzer_id': parse('$.result.analyzerId').find(data)[0].value,
                    'api_version': parse('$.result.apiVersion').find(data)[0].value,
                    'createdAt': parse('$.result.createdAt').find(data)[0].value,
                    'Customer_Questions': match.value
                }
                data_list.append(row)

# Create a DataFrame from the list
df = pd.DataFrame(data_list, columns=['id', 'Filename', 'analyzer_id', 'api_version', 'createdAt', 'Customer_Questions'])

# Display the DataFrame
display(df)

In [ ]:
# Save the DataFrame to a CSV file in extract folder with timestamp as suffix in filename
now = datetime.datetime.now()
timestamp = now.strftime("%Y%m%d%H%M%S")
df.to_csv(f'{folder_path}/extract_{timestamp}.csv', index=False)
print(f"Saved extracted data to {folder_path}/extract_{timestamp}.csv")
